In [1]:
import pandas as pd
import geopandas as gpd
import numpy as np

In [2]:
shapefiles_path = "../../data/shapefiles/"

# 1. Shapefiles

## 1.1 World Bank Shapefiles - Loading, Merging and Cleaning 

In [3]:
# List of Mashreq countries
country_names = ["Iraq", "Jordan", "Lebanon", "Syrian Arab Republic", "West Bank and Gaza"]

In [4]:
# Load the World Bank shapefile for admin level 0
wb_adm0_shp = gpd.read_file(shapefiles_path + "/WB_GAD_Med_v2/WB_GAD_ADM0.shp", encoding='utf-8')

# Filter for country names in the list of Mashreq countries
wb_adm0_shp = wb_adm0_shp.loc[wb_adm0_shp["NAM_0"].isin(country_names)].reset_index(drop=True)

# Drop unnecessary columns
wb_adm0_shp.drop(columns=['MIN_X', 'MIN_Y', 'MAX_X', 'MAX_Y', 'CENTER_X', 'CENTER_Y'], inplace=True)

# Add column indicating admin level
wb_adm0_shp["admin_level"] = 0

In [5]:
# Load the World Bank shapefile for admin level 1
wb_adm1_shp = gpd.read_file(shapefiles_path + "/WB_GAD_Med_v2/WB_GAD_ADM1.shp", encoding='utf-8')

# Filter for country names in the list of Mashreq countries
wb_adm1_shp = wb_adm1_shp.loc[wb_adm1_shp["NAM_0"].isin(country_names)].reset_index(drop=True)

# Add column indicating admin level
wb_adm1_shp["admin_level"] = 1

/home/dominik/Documents/wb2024/implementations/food_crises_mashreq/exploratory_news_sources/.venv/lib/python3.10/site-packages/pyogrio/raw.py:196: RuntimeWarning: ../../data/shapefiles//WB_GAD_Med_v2/WB_GAD_ADM1.shp contains polygon(s) with rings with invalid winding order. Autocorrecting them, but that shapefile should be corrected using ogr2ogr for example.
  return ogr_read(


In [6]:
# Load the World Bank shapefile for admin level 2
wb_adm2_shp = gpd.read_file(shapefiles_path + "/WB_GAD_Med_v2/WB_GAD_ADM2.shp", encoding='utf-8')

# Filter for country names in the list of Mashreq countries
wb_adm2_shp = wb_adm2_shp.loc[wb_adm2_shp["NAM_0"].isin(country_names)].reset_index(drop=True)

# Drop unnecessary columns
wb_adm2_shp.drop(columns=['OBJECTID','Shape_Leng', 'Shape_Le_1', 'Shape_Le_2', 'Shape_Area', 'ADM_LEVEL', 'GAD_ID_0', 'GAD_ID_1', 'NAM_0_Alt'], inplace=True)

# Exclude the admin level 2 for West Bank and Gaza
wb_adm2_shp = wb_adm2_shp.loc[wb_adm2_shp["NAM_0"] != "West Bank and Gaza"].reset_index(drop=True)

# Add column indicating admin level
wb_adm2_shp["admin_level"] = 2

/home/dominik/Documents/wb2024/implementations/food_crises_mashreq/exploratory_news_sources/.venv/lib/python3.10/site-packages/pyogrio/raw.py:196: RuntimeWarning: ../../data/shapefiles//WB_GAD_Med_v2/WB_GAD_ADM2.shp contains polygon(s) with rings with invalid winding order. Autocorrecting them, but that shapefile should be corrected using ogr2ogr for example.
  return ogr_read(


In [7]:
# Join all columns
wb_shp = pd.concat([wb_adm0_shp, wb_adm1_shp, wb_adm2_shp]).reset_index(drop=True)

# Drop columns that dont contain any data
wb_shp.fillna(value=np.nan, inplace=True)
wb_shp.drop(columns=wb_shp.columns[wb_shp.isnull().all()], inplace=True)

In [8]:
# Replace the country names
wb_shp.loc[wb_shp["NAM_0"] == "Syrian Arab Republic" , "NAM_0"] = "Syria"
wb_shp.loc[wb_shp["NAM_0"] == "West Bank and Gaza" , "NAM_0"] = "Palestine"

In [9]:
# Add Arabic names for the countries
arabic_country_names = {"Iraq": "العراق", 
                        "Jordan": "الأردن", 
                        "Lebanon": "لبنان", 
                        "Syrian Arab Republic": "سوريا", 
                        "Palestine": "فلسطين"}

wb_shp.loc[wb_shp["NAM_0"] == "Iraq" , "NAM_0_NTVE"] = arabic_country_names["Iraq"]
wb_shp.loc[wb_shp["NAM_0"] == "Jordan" , "NAM_0_NTVE"] = arabic_country_names["Jordan"]
wb_shp.loc[wb_shp["NAM_0"] == "Lebanon" , "NAM_0_NTVE"] = arabic_country_names["Lebanon"]
wb_shp.loc[wb_shp["NAM_0"] == "Syria" , "NAM_0_NTVE"] = arabic_country_names["Syrian Arab Republic"]
wb_shp.loc[wb_shp["NAM_0"] == "Palestine" , "NAM_0_NTVE"] = arabic_country_names["Palestine"]

In [10]:
wb_shp.drop(columns=["WB_A3", 'GAD_ID_0', 'GAD_ID_1', "WB_REGION", "WB_STATUS", "SOVEREIGN"], inplace=True)

## 1.2 Palestine - Cleaning and Translation

In [11]:
# How to handle West Bank and Gaza? Many News articles refer to Palestine

# Change admin level 1 to admin level 2
wb_shp.loc[(wb_shp["NAM_0"] == "Palestine") & (wb_shp["admin_level"] == 1) , "admin_level"] = 2
wb_shp.loc[wb_shp["NAM_0"] == "Palestine", 'NAM_2'] = wb_shp.loc[wb_shp["NAM_0"] == "Palestine", 'NAM_1']
wb_shp.loc[wb_shp["NAM_0"] == "Palestine", 'NAM_2_GAUL'] = wb_shp.loc[wb_shp["NAM_0"] == "Palestine", 'NAM_1_GAUL']
wb_shp.loc[wb_shp["NAM_0"] == "Palestine", 'NAM_1'] = np.nan
wb_shp.loc[wb_shp["NAM_0"] == "Palestine", 'NAM_1_GAUL'] = np.nan

# Create a copy of admin level 0 and split it into west bank and gaza
palestine_adm1 = wb_shp.loc[(wb_shp["NAM_0"] == "Palestine") & (wb_shp["admin_level"] == 0), ].copy()

palestine_adm1["admin_level"] = 1
palestine_adm1 = palestine_adm1.explode().reset_index(drop=True)

palestine_adm1.loc[0, "NAM_1"] = "West Bank"
palestine_adm1.loc[1, "NAM_1"] = "Gaza"

wb_shp = pd.concat([wb_shp, palestine_adm1]).reset_index(drop=True)

In [12]:
# Assign admin level 2 to admin level 1 locations
palestine_adm2_adm1_dict = {"Al Khalil (Hebron)" : "West Bank",
                            "Al Quds (Jerusalem)" : "West Bank",
                            "Ariha (Jericho)" : "West Bank",
                            "Bethlehem" : "West Bank",
                            "Deir al Balah" : "Gaza",
                            "Gaza" : "Gaza",
                            "Jabalya" : "Gaza",
                            "Jenin" : "West Bank",
                            "Khan Yunis" : "Gaza",
                            "Nablus" : "West Bank",
                            "Qalqiliya" : "West Bank",
                            "Rafah" : "Gaza",
                            "Ramallah" : "West Bank",
                            "Salfit" : "West Bank",
                            "Tubas" : "West Bank",
                            "Tulkarm" : "West Bank"}

wb_shp.loc[(wb_shp["NAM_0"] == "Palestine") & (wb_shp["admin_level"] == 2), "NAM_1"] = wb_shp.loc[(wb_shp["NAM_0"] == "Palestine") & (wb_shp["admin_level"] == 2), "NAM_2"].apply(lambda x: palestine_adm2_adm1_dict[x]) 

In [13]:
# Assign Arabic names to the "provinces"
palestine_provinces_arabic = {"Gaza" : "غزة",
                            "West Bank" : "الضفة الغربية"}

wb_shp.loc[wb_shp["NAM_0"] == "Palestine", "NAM_1_NTVE"] = wb_shp.loc[wb_shp["NAM_0"] == "Palestine", "NAM_1"].apply(lambda x: palestine_provinces_arabic[x] if x in palestine_provinces_arabic.keys() else x)

In [14]:
# Assign Arabic names to the districts
palestine_districts_arabic = {"Al Khalil (Hebron)" : "الخليل",
                            "Al Quds (Jerusalem)" : "القدس",
                            "Ariha (Jericho)" : "أريحا والأغوار",
                            "Bethlehem" : "بيت لحم",
                            "Deir al Balah" : "دير البلح",
                            "Gaza" : "غزة",
                            "Jabalya" : "شمال غزة",
                            "Jenin" : "جنين",
                            "Khan Yunis" : "خان يونس",
                            "Nablus" : "نابلس",
                            "Qalqiliya" : "قلقيلية",
                            "Rafah" : "رفح",
                            "Ramallah" : "رام الله والبيرة",
                            "Salfit" : "سلفيت",
                            "Tubas" : "طوباس والأغوار الشمالية",
                            "Tulkarm" : "طولكرم"}

wb_shp.loc[wb_shp["NAM_0"] == "Palestine", "NAM_2_NTVE"] = wb_shp.loc[wb_shp["NAM_0"] == "Palestine", "NAM_2"].apply(lambda x: palestine_districts_arabic[x] if x in palestine_districts_arabic.keys() else x)

In [15]:
# Exclude the part in the parentheses
wb_shp.loc[wb_shp["NAM_2"] == "Al Khalil (Hebron)" , "NAM_2"] = "Al Khalil"
wb_shp.loc[wb_shp["NAM_2"] == "Al Quds (Jerusalem)" , "NAM_2"] = "Al Quds"
wb_shp.loc[wb_shp["NAM_2"] == "Ariha (Jericho)" , "NAM_2"] = "Ariha"

## 1.3 Iraq - Translation

In [16]:
# Translate Provinces based on a shapefile from UNOCHA: 
iraq_provinces_arabic = {'Anbar': 'الانبار',
                        'Babil': 'بابل',
                        'Baghdad': 'بغداد',
                        'Basrah': 'البصرة',
                        'Dahuk': 'دهوك',
                        'Diyala': 'ديالى',
                        'Erbil': 'اربيل',
                        'Kerbala': 'كربلاء',
                        'Kirkuk': 'كركوك',
                        'Missan': 'ميسان',
                        'Muthanna': 'المثنى',
                        'Najaf': 'النجف',
                        'Ninewa': 'نينوى',
                        'Qadissiya': 'القادسية',
                        'Salah al-Din': 'صلاح الدين',
                        'Sulaymaniyah': 'السليمانية',
                        'Thi-Qar': 'ذي قار',
                        'Wassit': 'واسط'}

wb_shp.loc[wb_shp["NAM_0"] == "Iraq", "NAM_1_NTVE"] = wb_shp.loc[wb_shp["NAM_0"] == "Iraq", "NAM_1"].apply(lambda x: iraq_provinces_arabic[x] if x in iraq_provinces_arabic.keys() else x)

In [17]:
# Translate Districts based on a shapefile from UNOCHA: 
iraq_districts_arabic = {"Al-Ka'im": 'القائم',
                        'Al-Rutba': 'الرطبة',
                        'Ana': 'عنه',
                        'Falluja': 'الفلوجة',
                        'Haditha': 'حديثة',
                        'Heet': 'هيت',
                        "Ra'ua": np.nan,
                        'Ramadi': 'الرمادي',
                        'Al-Mahawil': 'المحاويل',
                        'Al-Musayab': 'المسيب',
                        'Hashimiya': 'الهاشمية',
                        'Hilla': 'الحلة',
                        'Abu Ghraib': np.nan,
                        'Adhamiya': 'العمادية',
                        'Al Resafa': 'الرصافة',
                        'Kadhmiyah': 'الكاظمية',
                        'Karkh': 'الكرخ',
                        "Mada'in": 'المدائن',
                        'Mahmoudiya': 'المحمودية',
                        'Tarmia': np.nan,
                        'Thawra 1': 'الثورة',
                        'Thawra 2': 'الثورة',
                        'Abu Al-Khaseeb': 'ابي الخصيب',
                        'Al-Midaina': 'المدينة',
                        'Al-Qurna': 'القرنة',
                        'Al-Zubair': 'الزبير',
                        'Basrah': 'البصرة',
                        'Fao': 'الفاو',
                        'Shatt Al-Arab': 'شط العرب',
                        'Amedi': np.nan,
                        'Dahuk': 'دهوك',
                        'Sumel': 'سميل',
                        'Zakho': 'زاخو',
                        'Al-Khalis': 'الخالص',
                        'Al-Muqdadiya': 'المقدادية',
                        "Ba'quba": 'بعقوبة',
                        'Baladrooz': 'مندلي',
                        'Khanaqin': 'خانقين',
                        'Kifri': 'كفري',
                        'Choman': np.nan,
                        'Erbil': 'اربيل',
                        'Koisnjaq': 'كويسنجق',
                        'Makhmur': 'مخمور',
                        'Mergasur': np.nan,
                        'Shaqlawa': 'شقلاوة',
                        'Soran': np.nan,
                        'Ain Al-Tamur': 'عين التمر',
                        'Al-Hindiya': 'الهندية',
                        'Kerbala': 'كربلاء',
                        'Al-Hawiga': 'الحويجة',
                        'Dabes': np.nan,
                        'Daquq': 'داقوق',
                        'Kirkuk': 'كركوك',
                        'Al-Kahla': 'الكحلاء',
                        'Al-Maimouna': 'الميمونة',
                        'Al-Mejar Al-Kabi': 'المجر الكبير',
                        'Ali Al-Gharbi': 'علي الغربي',
                        'Amara': 'العمارة',
                        "Qal'at Saleh": 'قلعة صالح',
                        'Al-Khidhir': 'الخضر',
                        'Al-Rumaitha': 'الرميثة',
                        'Al-Salman': 'السلمان',
                        'Al-Samawa': 'السماوة',
                        'Al-Manathera': 'المناذرة',
                        'Kufa': 'الكوفة',
                        'Najaf': 'النجف',
                        'Akre': 'عقرة',
                        "Al-Ba'aj": 'البعاج',
                        'Al-Hamdaniya': 'الحمدانية',
                        'Al-Shikhan': 'الشيخان',
                        'Hatra': 'الحضر',
                        'Mosul': 'الموصل',
                        'Sinjar': 'سنجار',
                        'Telafar': 'تلعفر',
                        'Tilkaif': 'تلكيف',
                        'Afaq': 'عفك',
                        'Al-Shamiya': 'الشامية',
                        'Diwaniya': 'الديوانية',
                        'Hamza': 'الحمزة',
                        'Al-Daur': 'الدور',
                        'Al-Fares': np.nan,
                        'Al-Shirqat': 'الشرقاط',
                        'Al-Thethar': np.nan,
                        'Baiji': 'بيجي',
                        'Balad': 'بلد',
                        'Samarra': 'سامراء',
                        'Tikrit': 'تكريت',
                        'Tooz': 'طوز خورماتو',
                        'Chamchamal': 'جمجمال',
                        'Darbandihkan': 'دربندخان',
                        'Dokan': 'دوكان',
                        'Halabja': 'حلبجة',
                        'Kalar': 'كلار',
                        'Penjwin': 'بنجوين',
                        'Pshdar': 'بشدر',
                        'Rania': 'رانية',
                        'Sharbazher': 'شهربازار',
                        'Sulaymaniya': 'السليمانية',
                        'Al-Chibayish': 'الجبايش',
                        "Al-Rifa'i": 'الرفاعي',
                        'Al-Shatra': 'الشطرة',
                        'Nassriya': 'الناصرية',
                        'Suq Al-Shoyokh': 'سوق الشيوخ',
                        'Al-Azezia': np.nan,
                        'Al-Hai': 'الحي',
                        "Al-Na'maniya": 'النعمانية',
                        'Al-Suwaira': 'الصويرة',
                        'Badra': 'بدرة',
                        'Kut': 'الكوت'}

wb_shp.loc[wb_shp["NAM_0"] == "Iraq", "NAM_2_NTVE"] = wb_shp.loc[wb_shp["NAM_0"] == "Iraq", "NAM_2"].apply(lambda x: iraq_districts_arabic[x] if x in iraq_districts_arabic.keys() else x)

## 1.4 Jordan - Translation

In [18]:
# Translate Provinces
jordan_provinces_arabic = {'Ajloon': 'عجلون',
                            'Amman': 'عمان',
                            'Aqaba': 'العقبة',
                            'Balqa': 'البلقاء',
                            'Irbid': 'إربد',
                            'Jarash': 'جرش',
                            'Karak': 'الكرك',
                            "Ma'an": 'معان',
                            'Madaba': 'مادبا',
                            'Mafraq': 'المفرق',
                            'Tafiela': 'الطفيلة',
                            'Zarqa': 'الزرقاء'}

wb_shp.loc[wb_shp["NAM_0"] == "Jordan", "NAM_1_NTVE"] = wb_shp.loc[wb_shp["NAM_0"] == "Jordan", "NAM_1"].apply(lambda x: jordan_provinces_arabic[x] if x in jordan_provinces_arabic.keys() else x)

In [19]:
# Translate Districts
jordan_districts_arabic = {'Ajloon': 'قصبة عجلون',
                            'Kufranjeh': 'كفرنجة',
                            'Al-Jeeza': 'الجيزة',
                            'Al-Muwaqar': 'الموقر',
                            'Amman': 'قصبة عمان',
                            "Jami'ah": 'الجامعة',
                            'Marka': 'ماركا',
                            "Na'oor": 'ناعور',
                            'Qwaismeh': 'القويسمة',
                            'Sahab': 'سحاب',
                            'Wadisseer': 'وادي السير',
                            'Al-Qwaira': 'القويرة',
                            'Aqaba': 'قصبة العقبة',
                            'Wadi Araba': np.nan,
                            'Al-Ardha': np.nan,
                            'Al-Ruseifa': 'الرصيفة',
                            'Al-Salt': 'قصبة السلط',
                            'Al-Shoona Al-Janoobiya': 'الشونة الجنوبية',
                            'Dair Ala': 'دير عال',
                            'Ira & Yarqa': np.nan,
                            'Mahes & Fhais': 'ماحص والفحيص',
                            'Zay': np.nan,
                            'Al-Aghwar Al-Shimaliya': 'االغوار الشمالية',
                            'Al-Koora': 'الكورة',
                            'Al-Mazar Al-Shimali': 'المزار الشمالي',
                            'Al-Ramtha': 'الرمثا',
                            'Al-Teeba': np.nan,
                            'Al-Wasatiya': 'الوسطية',
                            'Beni Kinana': 'بني كنانة',
                            'Beni Obaid': 'بني عبيد',
                            'Irbid': 'قصبة اربد',
                            'Jarash': 'قصبة جرش',
                            'Al-Aghwar Al-Janoobiya': 'االغوار الجنوبية',
                            'Al-Mazar Al-Janoobi': 'المزار الجنوبي',
                            'Al-Qaser': 'القصر',
                            'Al-Qatraneh': 'القطرانة',
                            'Ayy': 'عي',
                            "Faqooa'": 'فقوع',
                            'Karak': 'قصبة الكرك',
                            'Ail': np.nan,
                            'Al-Jafer': np.nan,
                            'Al-Mraigha': np.nan,
                            'Al-Shobak': 'الشوبك',
                            "Ma'an": 'قصبة معان',
                            'Wadi Moosa': np.nan,
                            'Madaba': 'قصبة مادبا',
                            'Theeban': np.nan,
                            'Al-ruwaished': 'الرويشد',
                            "Bal'ama": np.nan,
                            'Mafraq': 'المفرق',
                            'Sabha': 'سحاب',
                            'Sama Al-Serhan': '',
                            'Al-Hasa': 'الحسا',
                            'Bsaira': 'بصيرا',
                            'Tafiela': 'قصبة الطفيلة',
                            'Al-Dhilail': np.nan,
                            'Al-Hashimiya': 'الهاشمية',
                            'Zarqa': 'قصبة الزرقاء'}

wb_shp.loc[wb_shp["NAM_0"] == "Jordan", "NAM_2_NTVE"] = wb_shp.loc[wb_shp["NAM_0"] == "Jordan", "NAM_2"].apply(lambda x: jordan_districts_arabic[x] if x in jordan_districts_arabic.keys() else x)

## 1.5 English Names - Clean columns

In [20]:
# Transform all English names to lower case
wb_shp["NAM_0"] = wb_shp["NAM_0"].str.lower()

wb_shp["NAM_1"] = wb_shp["NAM_1"].str.lower()
wb_shp["NAM_1_STAT"] = wb_shp["NAM_1_STAT"].str.lower()
wb_shp["NAM_1_SRCE"] = wb_shp["NAM_1_SRCE"].str.lower()
wb_shp["NAM_1_GAUL"] = wb_shp["NAM_1_GAUL"].str.lower()
wb_shp["NAM_1_WIKI"] = wb_shp["NAM_1_WIKI"].str.lower()

wb_shp["NAM_2"] = wb_shp["NAM_2"].str.lower()
wb_shp["NAM_2_GAUL"] = wb_shp["NAM_2_GAUL"].str.lower()
wb_shp["NAM_2_STAT"] = wb_shp["NAM_2_STAT"].str.lower()
wb_shp["NAM_2_SRCE"] = wb_shp["NAM_2_SRCE"].str.lower()
wb_shp["NAM_2_WIKI"] = wb_shp["NAM_2_WIKI"].str.lower()

In [21]:
# Create function that removes prefixes for Arabic articles from the given list of words
def remove_prefix(list):
    """
    Converts all words in the given list to lowercase and removes Arabic article prefixes such as "al-", "al ", "ar-", "ar ", "as-", "as ", "el-", "el ", "ath-", "ath ", "az-", and "az ".
    
    Args:
        list (list): The list of words to be processed.
    
    Returns:
        list: The processed list of words with lowercase and cleaned prefixes.
    """
    
    return [word.removeprefix("al-").removeprefix("al ").removeprefix("ar-").removeprefix("ar ").removeprefix("as-").removeprefix("as ").removeprefix("el-").removeprefix("el ").removeprefix("ath-").removeprefix("ath ").removeprefix("az-").removeprefix("az ").removeprefix("an-").removeprefix("an ") for word in list]

### 1.5.1 Admin Level 1

There are four columns with names for locations on Admin level 1: NAM_1_GAUL, NAM_1_STAT, NAM_1_SRCE, NAM_1_WIKI.

In most cases only one or two of these columns contain names. In other cases multiple columns contain the same name. 
The following code identifies the unique names for each location and saves them in two columns called "NAM_1_AL1" and "NAM_1_AL2".

In [22]:
# Remove the prefixes from the admin level 1 location names
wb_shp.loc[~wb_shp["NAM_1"].isna(), "NAM_1"] = remove_prefix(wb_shp.loc[~wb_shp["NAM_1"].isna(), "NAM_1"].values)
wb_shp.loc[~wb_shp["NAM_1_STAT"].isna(), "NAM_1_STAT"] = remove_prefix(wb_shp.loc[~wb_shp["NAM_1_STAT"].isna(), "NAM_1_STAT"].values)
wb_shp.loc[~wb_shp["NAM_1_SRCE"].isna(), "NAM_1_SRCE"] = remove_prefix(wb_shp.loc[~wb_shp["NAM_1_SRCE"].isna(), "NAM_1_SRCE"].values)

In [23]:
# Check if columns contain the same value
wb_shp.loc[wb_shp["NAM_1"] == wb_shp["NAM_1_GAUL"], "NAM_1_GAUL"] = np.nan
wb_shp.loc[wb_shp["NAM_1"] == wb_shp["NAM_1_STAT"], "NAM_1_STAT"] = np.nan
wb_shp.loc[wb_shp["NAM_1"] == wb_shp["NAM_1_SRCE"], "NAM_1_SRCE"] = np.nan
wb_shp.loc[wb_shp["NAM_1"] == wb_shp["NAM_1_WIKI"], "NAM_1_WIKI"] = np.nan

In [24]:
wb_shp.loc[wb_shp["NAM_1_STAT"].isna(), "NAM_1_STAT"] = wb_shp.loc[wb_shp["NAM_1_STAT"].isna(), "NAM_1_SRCE"]
wb_shp.loc[wb_shp["NAM_1_SRCE"] == wb_shp["NAM_1_STAT"], "NAM_1_SRCE"] = np.nan
wb_shp.drop(columns=["NAM_1_GAUL", "NAM_1_SRCE"], inplace=True)

In [25]:
wb_shp.rename(columns={"NAM_1_STAT": "NAM_1_AL1", "NAM_1_WIKI": "NAM_1_AL2"}, inplace=True)

### 1.5.2 Admin Level 2

There are four columns with names for locations on Admin level 2: NAM_2_GAUL, NAM_2_STAT, NAM_2_SRCE, NAM_2_WIKI.

In most cases only one or two of these columns contain names. In other cases multiple columns contain the same name. 
The following code identifies the unique names for each location and saves them in two columns called "NAM_2_AL1" and "NAM_2_AL2".

In [26]:
# Remove the prefixes from the admin level 2 location names
wb_shp.loc[~wb_shp["NAM_2"].isna(), "NAM_2"] = remove_prefix(wb_shp.loc[~wb_shp["NAM_2"].isna(), "NAM_2"].values)
wb_shp.loc[~wb_shp["NAM_2_STAT"].isna(), "NAM_2_STAT"] = remove_prefix(wb_shp.loc[~wb_shp["NAM_2_STAT"].isna(), "NAM_2_STAT"].values)
wb_shp.loc[~wb_shp["NAM_2_SRCE"].isna(), "NAM_2_SRCE"] = remove_prefix(wb_shp.loc[~wb_shp["NAM_2_SRCE"].isna(), "NAM_2_SRCE"].values)

In [27]:
# Check if columns contain the same value
wb_shp.loc[wb_shp["NAM_2"] == wb_shp["NAM_2_GAUL"], "NAM_2_GAUL"] = np.nan
wb_shp.loc[wb_shp["NAM_2"] == wb_shp["NAM_2_STAT"], "NAM_2_STAT"] = np.nan
wb_shp.loc[wb_shp["NAM_2"] == wb_shp["NAM_2_SRCE"], "NAM_2_SRCE"] = np.nan
wb_shp.loc[wb_shp["NAM_2"] == wb_shp["NAM_2_WIKI"], "NAM_2_WIKI"] = np.nan

In [28]:
# Drop column only containing np.nan
wb_shp.drop(columns=["NAM_2_STAT"], inplace=True)

In [29]:
# Check if columns contain the same value
wb_shp.loc[wb_shp["NAM_2_GAUL"].isna(), "NAM_2_GAUL"] = wb_shp.loc[wb_shp["NAM_2_GAUL"].isna(), "NAM_2_SRCE"]
wb_shp.loc[wb_shp["NAM_2_GAUL"] == wb_shp["NAM_2_SRCE"], "NAM_2_SRCE"] = np.nan

wb_shp.loc[wb_shp["NAM_2_GAUL"].isna(), "NAM_2_GAUL"] = wb_shp.loc[wb_shp["NAM_2_GAUL"].isna(), "NAM_2_WIKI"]
wb_shp.loc[wb_shp["NAM_2_GAUL"] == wb_shp["NAM_2_WIKI"], "NAM_2_WIKI"] = np.nan

wb_shp.loc[wb_shp["NAM_2_SRCE"] == wb_shp["NAM_2_WIKI"], "NAM_2_WIKI"] = np.nan

In [30]:
# Drop column only containing np.nan
wb_shp.drop(columns=["NAM_2_SRCE"], inplace=True)

In [31]:
# Rename columns
wb_shp.rename(columns={"NAM_2_GAUL": "NAM_2_AL1", "NAM_2_WIKI": "NAM_2_AL2"}, inplace=True)

## 1.6 Creating IDs

In [32]:
# Creating an ID column for Admin level 0 and Admin level 1
wb_shp["ID_0"] = wb_shp["HASC_0"]

wb_shp["ID_1"] = wb_shp["HASC_1"].str.replace(".", "_")

In [33]:
# Adding the ID for Gaza and West Bank
wb_shp.loc[wb_shp["NAM_1"] == "gaza", "ID_1"] = "PS_GZ"
wb_shp.loc[wb_shp["NAM_1"] == "west bank", "ID_1"] = "PS_WB"

In [34]:
# Create an ID for the districts by counting within each province starting from 1
wb_shp.loc[wb_shp["admin_level"] == 2, "ID_2"] = wb_shp.loc[wb_shp["admin_level"] == 2].groupby('ID_1').cumcount() + 1
wb_shp['ID_2'] = wb_shp['ID_2'].astype(str).apply(lambda x: x.split(".")[0])

# Combine Group and ID to create the desired format
wb_shp['ID_2'] = wb_shp['ID_1'] + '_' + wb_shp['ID_2']

In [35]:
# For columns corresponding to admin level 1, set ID_2 to np.nan
wb_shp.loc[wb_shp["admin_level"] == 1, "ID_2"] = np.nan

In [36]:
# Putting all IDs in lower case
wb_shp["ID_0"] = wb_shp["ID_0"].str.lower()
wb_shp["ID_1"] = wb_shp["ID_1"].str.lower()
wb_shp["ID_2"] = wb_shp["ID_2"].str.lower()

## 1.7 Creating one Name and ID column

In [37]:
# Create column containing the English name of the location
wb_shp["NAME"] = ""
wb_shp.loc[wb_shp["admin_level"] == 0, "NAME"] = wb_shp.loc[wb_shp["admin_level"] == 0, "NAM_0"].values
wb_shp.loc[wb_shp["admin_level"] == 1, "NAME"] = wb_shp.loc[wb_shp["admin_level"] == 1, "NAM_1"].values
wb_shp.loc[wb_shp["admin_level"] == 2, "NAME"] = wb_shp.loc[wb_shp["admin_level"] == 2, "NAM_2"].values

# Create column containing the Arabic name of the location
wb_shp["NAME_NTVE"] = ""
wb_shp.loc[wb_shp["admin_level"] == 0, "NAME_NTVE"] = wb_shp.loc[wb_shp["admin_level"] == 0, "NAM_0_NTVE"].values
wb_shp.loc[wb_shp["admin_level"] == 1, "NAME_NTVE"] = wb_shp.loc[wb_shp["admin_level"] == 1, "NAM_1_NTVE"].values
wb_shp.loc[wb_shp["admin_level"] == 2, "NAME_NTVE"] = wb_shp.loc[wb_shp["admin_level"] == 2, "NAM_2_NTVE"].values

# Create column containing the ID of the location
wb_shp["ID"] = ""
wb_shp.loc[wb_shp["admin_level"] == 0, "ID"] = wb_shp.loc[wb_shp["admin_level"] == 0, "ID_0"].values
wb_shp.loc[wb_shp["admin_level"] == 1, "ID"] = wb_shp.loc[wb_shp["admin_level"] == 1, "ID_1"].values
wb_shp.loc[wb_shp["admin_level"] == 2, "ID"] = wb_shp.loc[wb_shp["admin_level"] == 2, "ID_2"].values

In [38]:
# Select columns to keep
wb_shp = wb_shp[['admin_level', 
                'ID', 'NAME', 'NAME_NTVE',
                'ID_0', 'ID_1', 'ID_2',
                'NAM_0', 
                'NAM_1', "NAM_1_AL1", "NAM_1_AL2",
                'NAM_2', "NAM_2_AL1", "NAM_2_AL2",
                'NAM_0_NTVE', 'NAM_1_NTVE', 'NAM_2_NTVE',
                'DATA_SRC', 'DATA_DATE','LAST_UPDTE', 'geometry']]

In [39]:
# Rename admin level column because shapefile column names can be max 10 character
wb_shp.rename(columns={"admin_level": "adml"}, inplace=True)

In [40]:
wb_shp.to_file(shapefiles_path + "WB_clean/WB_clean.shp", index=False)

# Splitting the shapefile into different shapefiles for levels

In [ ]:
wb_shp = gpd.read_file(shapefiles_path + "WB_clean/WB_clean.shp", encoding='utf-8')

In [ ]:
wb_shp["NAME"] = wb_shp["NAME"].apply(lambda x: " ".join([val.capitalize() for val in x.split(" ")]))
wb_shp["NAM_0"] = wb_shp["NAM_0"].apply(lambda x: " ".join([val.capitalize() for val in x.split(" ")]))
wb_shp.loc[~wb_shp["NAM_1"].isna(), "NAM_1"] = wb_shp.loc[~wb_shp["NAM_1"].isna(), "NAM_1"].apply(lambda x: " ".join([val.capitalize() for val in x.split(" ")]))
wb_shp.loc[~wb_shp["NAM_2"].isna(), "NAM_2"] = wb_shp.loc[~wb_shp["NAM_2"].isna(), "NAM_2"].apply(lambda x: " ".join([val.capitalize() for val in x.split(" ")]))

In [ ]:
wb_shp_adm0 = wb_shp.loc[wb_shp["adml"] == 0].reset_index(drop=True)
wb_shp_adm1 = wb_shp.loc[wb_shp["adml"] == 1].reset_index(drop=True)
wb_shp_adm2 = wb_shp.loc[wb_shp["adml"] == 2].reset_index(drop=True)

In [ ]:
wb_shp_adm0.drop(columns=["ID", "ID_1", "ID_2"], inplace=True)
wb_shp_adm1.drop(columns=["ID", "ID_0", "ID_2"], inplace=True)
wb_shp_adm2.drop(columns=["ID", "ID_0", "ID_1"], inplace=True)

In [ ]:
wb_shp.to_file(shapefiles_path + "WB_clean/WB_clean.shp", encoding='utf-8')

wb_shp_adm0.to_file(shapefiles_path + "WB_clean/WB_clean_adm0.shp", encoding='utf-8')
wb_shp_adm1.to_file(shapefiles_path + "WB_clean/WB_clean_adm1.shp", encoding='utf-8')
wb_shp_adm2.to_file(shapefiles_path + "WB_clean/WB_clean_adm2.shp", encoding='utf-8')

In [ ]:
wb_shp.drop(columns=["geometry"], inplace=True)

wb_shp_adm0.drop(columns=["geometry"], inplace=True)
wb_shp_adm1.drop(columns=["geometry"], inplace=True)
wb_shp_adm2.drop(columns=["geometry"], inplace=True)

wb_shp_adm0.to_csv(shapefiles_path + "WB_clean/WB_clean_adm0.csv", index=False, encoding='utf-8')
wb_shp_adm1.to_csv(shapefiles_path + "WB_clean/WB_clean_adm1.csv", index=False, encoding='utf-8')
wb_shp_adm2.to_csv(shapefiles_path + "WB_clean/WB_clean_adm2.csv", index=False, encoding='utf-8')

wb_shp.to_csv(shapefiles_path + "WB_clean/WB_clean.csv", index=False, encoding='utf-8')